# 05b 동작 확인용 테스트

- 게임 3개 / 게임당 최대 300개 리뷰 (3페이지)
- 예상 소요: 1~2분
- 컬럼 구조·결측치·타입 확인 목적

In [ ]:
import requests
import pandas as pd
import time

# 테스트 대상: 고인기(Terraria) / 중간(Stardew Valley) / 저인기 상대적으로 작은 게임
TEST_GAMES = [
    {"appid": 105600,  "name": "Terraria",      "genre": "Action"},
    {"appid": 413150,  "name": "Stardew Valley", "genre": "Casual/Lightweight"},
    {"appid": 292030,  "name": "The Witcher 3",  "genre": "RPG"},
]

SLEEP_SEC            = 1.0
MAX_REVIEWS_PER_GAME = 300   # 3페이지만
MAX_RETRIES          = 3

print("설정 완료")

In [ ]:
def collect_game_reviews(appid, max_reviews=MAX_REVIEWS_PER_GAME):
    collected = []
    cursor = "*"

    while len(collected) < max_reviews:
        params = {
            "json": 1,
            "filter": "recent",
            "language": "all",
            "review_type": "all",
            "purchase_type": "all",
            "num_per_page": 100,
            "cursor": cursor,
            "filter_offtopic_activity": 1,
        }

        data = None
        for attempt in range(MAX_RETRIES):
            try:
                resp = requests.get(
                    f"https://store.steampowered.com/appreviews/{appid}",
                    params=params,
                    timeout=20,
                )
                resp.raise_for_status()
                data = resp.json()
                break
            except Exception as e:
                if attempt == MAX_RETRIES - 1:
                    print(f"  ⚠️ 재시도 초과: {e}")
                    return collected
                time.sleep(2 ** (attempt + 1))

        reviews = data.get("reviews", [])
        if not reviews:
            break

        new_cursor = data.get("cursor", "")
        if not new_cursor or new_cursor == cursor:
            break
        cursor = new_cursor

        for r in reviews:
            author = r.get("author", {})
            text   = r.get("review", "")
            collected.append({
                "review_id"             : r.get("recommendationid"),
                "timestamp_created"     : r.get("timestamp_created"),
                "voted_up"              : r.get("voted_up"),
                "playtime_at_review_min": author.get("playtime_at_review"),
                "playtime_forever_min"  : author.get("playtime_forever"),
                "language"              : r.get("language"),
                "review_length"         : len(text) if text else 0,
            })

        time.sleep(SLEEP_SEC)

    return collected


# 수집 실행
all_rows = []

for game in TEST_GAMES:
    appid = game["appid"]
    print(f"{game['name']} ({appid}) 수집 중...", flush=True)

    reviews = collect_game_reviews(appid)

    for r in reviews:
        r["app_id"]   = appid
        r["app_name"] = game["name"]
        r["genre"]    = game["genre"]
    all_rows.extend(reviews)

    print(f"  → {len(reviews)}개 수집")

df = pd.DataFrame(all_rows)
print(f"\n총 {len(df)}행 수집 완료")

In [ ]:
# 결과 확인
print("=== 컬럼 및 타입 ===")
print(df.dtypes)
print()

print("=== 결측치 ===")
print(df[["playtime_at_review_min", "playtime_forever_min", "voted_up"]].isna().sum())
print()

print("=== 샘플 3행 ===")
df.head(3)